# HumanVibe Gemini Prime — Direct API Runtime
This notebook is the executable Gemini runtime. It uses Google's official `google-genai` SDK and an explicit Gemini model. It does not claim to be the consumer Gemini web/app session.


In [ ]:
%pip install -q -U google-genai


In [ ]:
import os
import importlib.metadata
from pathlib import Path
from google import genai

MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY is missing")

def read_text(path, limit=None):
    p = Path(path)
    if not p.exists():
        return ""
    text = p.read_text(encoding="utf-8", errors="replace")
    return text if limit is None else text[-limit:]

rules = read_text("OPERATING_RULES.md")
handoff = read_text("HANDOFF_PROTOCOL.md")
chat_tail = read_text("CHAT_LOG.md", 18000)
training = read_text("TRAINING_MATRIX.md")
wake = read_text("GEMINI_WAKE.md")

# Keep the runtime bounded while preserving the canonical lessons most relevant
# to execution discipline and the Gemini identity failure.
wanted = ("HV-EXP-027", "HV-EXP-044", "HV-EXP-047", "HV-EXP-048")
training_parts = []
for key in wanted:
    pos = training.find(key)
    if pos >= 0:
        training_parts.append(training[pos:pos+5000])
training_context = "\n\n".join(training_parts)

prompt = f"""
You are the HumanVibe Gemini Prime NOTEBOOK runtime.

IDENTITY BOUNDARY:
- You are an official Gemini API invocation executed from a Jupyter notebook.
- You are NOT the consumer Gemini web/app chat session.
- Never claim access, memory, browser sessions, connectors, credentials, or authority that are not present in the supplied evidence.

CANONICAL OPERATING RULES:
{rules}

HANDOFF PROTOCOL:
{handoff}

RELEVANT TRAINING:
{training_context}

RECENT CHAT LOG:
{chat_tail}

CURRENT TEAMCHAT WAKE / ISSUE CONTEXT:
{wake}

TASK:
Answer the actual Teamchat task from CURRENT TEAMCHAT WAKE / ISSUE CONTEXT.
Apply the canonical rules, proof-before-claim, shortest-valid-path, and stop at the first genuine capability/authentication blocker.
Do not fabricate execution. Do not claim an external effect without supplied evidence.
Keep the answer concise.

Begin exactly:
TRAINING_REVIEW: PASS
STATUS: PASS|PARTIAL|BLOCKED
"""

client = genai.Client(api_key=API_KEY)
response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
)

answer = (response.text or "").strip()
model_version = getattr(response, "model_version", None)
sdk_version = importlib.metadata.version("google-genai")

evidence = "\n".join([
    "[PRIME_NOTEBOOK] DIRECT_GEMINI_API",
    f"SDK: google-genai {sdk_version}",
    f"MODEL_REQUESTED: {MODEL}",
    f"MODEL_VERSION: {model_version or 'not_returned'}",
    "SOURCE: Jupyter notebook -> official Google GenAI SDK -> Gemini API",
    "",
    answer,
    ""
])
Path("GEMINI_RESPONSE.md").write_text(evidence, encoding="utf-8")
print(evidence)
